# Code mẫu — Hyperparameter tuning

Tuning là chọn cả cấu hình của model lẫn cách xử lý dữ liệu đi kèm. Các ví dụ dưới đây bắt đầu từ việc đếm xem một vùng thử nhỏ tốn bao nhiêu lần fit, rồi chạy cross-validation với cả Pipeline, và lúc nào cũng giữ tập test đứng ngoài mọi bước lựa chọn.

## Ví dụ 1: Đếm ngân sách, và loại trước những cấu hình không chạy được

Một grid gồm ba giá trị k và hai kiểu trọng số của kNN. Với ba fold và một lần fit lại ở cuối, ta cần bao nhiêu lần fit? Ngoài ra, train ở đây chỉ có bốn mẫu, nên cấu hình k=5 thậm chí không dự đoán được. Ta cũng so k, một giá trị đặt từ trước, với hệ số của một model tuyến tính, một giá trị được học từ dữ liệu.

In [ ]:
import numpy as np
from sklearn.model_selection import ParameterGrid
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

grid = {"n_neighbors": [1, 3, 5], "weights": ["uniform", "distance"]}
configs = list(ParameterGrid(grid))
print("Số cấu hình:", len(configs))
print("Số fit với 3 fold và refit:", 3 * len(configs) + 1)

X_train = np.array([[0.0], [1.0], [2.0], [3.0]])
y_train = np.array([1.0, 3.0, 5.0, 7.0])
linear = LinearRegression().fit(X_train, y_train)
print("Hệ số được fit học:", np.round(linear.coef_, 2).tolist())
bad_k = 5
try:
    if bad_k > len(X_train):
        raise ValueError(f"k={bad_k} vượt {len(X_train)} mẫu train")
    KNeighborsRegressor(n_neighbors=bad_k).fit(X_train, y_train).predict([[1.5]])
except ValueError as error:
    print("Kiểm trước khi chạy:", error)

Số cấu hình: 6
Số fit với 3 fold và refit: 19
Hệ số được fit học: [2.0]
Kiểm trước khi chạy: k=5 vượt 4 mẫu train


Sáu cấu hình cần `6 × 3 + 1 = 19` lần fit. Trước khi chạy một vùng thử lớn, đếm như vậy giúp biết nó có vừa với thời gian của buổi thi không. Hệ số 2 là thứ LinearRegression tự học từ train, còn k là thứ ta phải đặt trước, rồi dùng validation để chọn. Đó chính là khác biệt giữa tham số và hyperparameter. Khi chạy cross-validation, phải so k với **train nhỏ nhất trong các fold**, chứ không phải với cả tập dev. Kiểm sớm như vậy giúp không tốn công cho một cấu hình chắc chắn hỏng.

## Ví dụ 2: Grid search, với cả Pipeline chạy lại trong mỗi fold

Iris có 150 bông hoa, bốn số đo của đài và cánh hoa tính bằng cm, và ba loài. Ta tách 30 bông làm test ngay từ đầu, và chỉ dùng 120 bông còn lại, gọi là tập dev, để so các giá trị k và weights theo macro-F1. Scaler phải nằm trong Pipeline, vì khoảng cách của kNN phụ thuộc vào thang đo của từng cột.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix

X, y = load_iris(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=30, stratify=y, random_state=71,
)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=71).split(X_dev, y_dev))
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
parameters = {"knn__n_neighbors": [1, 5, 15], "knn__weights": ["uniform", "distance"]}
assert max(parameters["knn__n_neighbors"]) <= min(len(train_ids) for train_ids, _ in folds)
search = GridSearchCV(
    pipeline, parameters, scoring="f1_macro", cv=folds,
    return_train_score=True, refit=True, error_score="raise", n_jobs=1,
)
search.fit(X_dev, y_dev)
log = search.cv_results_
for params, train, valid in zip(log["params"], log["mean_train_score"], log["mean_test_score"]):
    print(f"k={params['knn__n_neighbors']:2d}, {params['knn__weights']:8s}: train F1={train:.3f}, CV F1={valid:.3f}")
print("Cấu hình chốt bằng CV:", search.best_params_)

k= 1, uniform : train F1=1.000, CV F1=0.917
k= 1, distance: train F1=1.000, CV F1=0.917
k= 5, uniform : train F1=0.958, CV F1=0.924
k= 5, distance: train F1=1.000, CV F1=0.942
k=15, uniform : train F1=0.937, CV F1=0.958
k=15, distance: train F1=1.000, CV F1=0.950
Cấu hình chốt bằng CV: {'knn__n_neighbors': 15, 'knn__weights': 'uniform'}


Cột mà scikit-learn đặt tên `mean_test_score` thật ra là điểm validation trung bình của các fold, chứ tập test 30 bông vẫn chưa được chấm. Cái tên dễ gây nhầm này đáng nhớ. Cấu hình `distance` có train F1 bằng 1 ở mọi k, vì khi dự đoán trên train, mỗi bông tự là hàng xóm của chính nó với khoảng cách 0. Vì vậy phải chọn theo cột CV. Nếu có nhiều cấu hình hòa điểm, GridSearchCV lấy cấu hình đứng trước trong thứ tự của grid.

Với `refit=True`, cấu hình thắng đã được fit lại trên cả 120 bông dev. Bây giờ mới đem đúng model ấy ra chấm trên test, và đọc lỗi qua confusion matrix.

In [ ]:
prediction = search.best_estimator_.predict(X_test)
print(f"Test macro-F1={f1_score(y_test, prediction, average='macro'):.3f}")
print(confusion_matrix(y_test, prediction, labels=[0, 1, 2]))
print("Scaler refit dùng số mẫu:", int(search.best_estimator_.named_steps["scale"].n_samples_seen_))

Test macro-F1=1.000
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]
Scaler refit dùng số mẫu: 120


Scaler cuối cùng được học trên 120 bông dev, còn các scaler trong lúc cross-validation thì mỗi cái được học riêng trên phần train của fold. Ma trận không có ô nào ngoài đường chéo, tức cả 30 bông test lần này đều được đoán đúng. Dù vậy, F1 bằng 1 trên một tập nhỏ chưa chứng minh model luôn đúng. Và nếu thấy kết quả này rồi quay lại sửa cấu hình, tập test sẽ biến thành dữ liệu để chọn cho vòng sau, không còn là test nữa.

## Ví dụ 3: Random search thử ít hơn trên cùng vùng

Vẫn là Iris, nhưng dựng lại từ đầu thành một thí nghiệm riêng. Giữ nguyên cách chia, các fold, metric, và vùng sáu cấu hình như trên. Grid search thử cả sáu, còn random search chỉ bốc ra ba. Ta chọn giữa hai kết quả theo điểm CV, hòa thì lấy grid, rồi mới mở test.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

X, y = load_iris(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=30, stratify=y, random_state=72,
)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=72).split(X_dev, y_dev))
pipeline = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier())])
region = {"knn__n_neighbors": [1, 5, 15], "knn__weights": ["uniform", "distance"]}
grid = GridSearchCV(pipeline, region, scoring="f1_macro", cv=folds, n_jobs=1, error_score="raise")
random = RandomizedSearchCV(
    pipeline, region, n_iter=3, scoring="f1_macro", cv=folds,
    random_state=72, n_jobs=1, error_score="raise",
)
for name, search in [("grid", grid), ("random", random)]:
    search.fit(X_dev, y_dev)
    candidates = len(search.cv_results_["params"])
    print(f"{name}: ứng viên={candidates}, fit={3*candidates+1}, best CV F1={search.best_score_:.3f}")
    print("  Chọn:", search.best_params_)
chosen_name, chosen = ("grid", grid) if grid.best_score_ >= random.best_score_ else ("random", random)
print("Chốt trước test:", chosen_name)
prediction = chosen.best_estimator_.predict(X_test)
print(f"Test macro-F1={f1_score(y_test, prediction, average='macro'):.3f}")

grid: ứng viên=6, fit=19, best CV F1=0.933
  Chọn: {'knn__n_neighbors': 5, 'knn__weights': 'distance'}
random: ứng viên=3, fit=10, best CV F1=0.932
  Chọn: {'knn__weights': 'distance', 'knn__n_neighbors': 15}
Chốt trước test: grid
Test macro-F1=1.000


Grid cần 19 lần fit, random chỉ cần 10, và hai bên dùng chung vùng thử cùng các fold nên so được với nhau. Random chỉ thử một phần của chính grid này, nên điểm CV tốt nhất của nó không thể vượt grid, và ở đây nó kém đúng 0.001. Lợi thế của random search là gặp được cấu hình khá tốt với ít lần fit hơn, nhất là khi vùng thử lớn và chỉ vài hyperparameter thật sự quan trọng. Một ví dụ nhỏ như thế này chưa chứng minh được random luôn thắng. Test F1 bằng 1 cũng chỉ mô tả 30 bông đã giữ riêng trong thí nghiệm này.